# Lab | Chains in LangChain

## Outline

* LLMChain
* Sequential Chains
  * SimpleSequentialChain
  * SequentialChain
* Router Chain

In [31]:
import os
import warnings
from google.colab import userdata
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

warnings.filterwarnings("ignore")
_ = load_dotenv(find_dotenv()) # This line loads environment variables from a .env file if it exists
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY missing"
print("Environment ready ✅")

Environment ready ✅


In [32]:
import pandas as pd
df = pd.read_csv('/content/Data.csv')

In [33]:
df.head()

,Product,Review
0,Queen Size Sheet Set,I ordered a king size set. My only criticism w...
1,Waterproof Phone Pouch,"I loved the waterproof sac, although the openi..."
2,Luxury Air Mattress,This mattress had a small hole in the top of i...
3,Pillows Insert,This is the best throw pillow fillers on Amazo...
4,Milk Frother Handheld\n,I loved this product. But they only seem to l...


## LLMChain

In [34]:
!pip install langchain langchain-core langchain-openai langchain-community langchain-classic

In [35]:
!pip show langchain langchain-core langchain-openai langchain-community

Name: langchain
Version: 1.3.13
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
---
Name: langchain-core
Version: 1.4.9
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: jsonpatch, langchain-protocol, langsmith, packaging, pydantic, pyyaml, tenacity, typing-extensions, uuid-utils
Required-by: langchain, langchain-classic, langchain-community, langchain-openai, langchain-text-splitters, langgraph, langgraph-checkpoint, langgraph-prebuilt, langgraph-sdk
---
Name: langchain-openai
Version: 1.3.5
Summary: An integration package connecting OpenAI and LangChain
Home-page: https://docs.langchain.com/oss/python/integrations/providers/openai
Author: 


In [36]:
from langchain_openai import ChatOpenAI        # ✅ 1.0.3
from langchain_core.prompts import ChatPromptTemplate  # ✅ From langchain-core 1.0.5
from langchain_classic.chains import LLMChain  # ✅ Legacy (install if missing)

In [42]:
# Retrieve the API key from environment variables
openai_api_key_value = os.getenv("OPENAI_API_KEY")

llm = ChatOpenAI(temperature=0, openai_api_key=openai_api_key_value)

In [43]:
prompt = ChatPromptTemplate.from_template( 'Provide detailed information about the product: {product}'
)

In [44]:
chain = LLMChain(llm=llm, prompt=prompt)

In [52]:
product = 'Waterproof Phone Pouch'
chain.run(product)

"A waterproof phone pouch is a protective case designed to keep your phone safe from water damage. It is typically made from a durable, waterproof material such as PVC or TPU that seals tightly to prevent water from seeping in. \n\nThese pouches are transparent, allowing you to still use your phone's touchscreen and camera while it is inside. They often come with a lanyard or strap for easy carrying, and some even have additional features such as a built-in compass or floatation device.\n\nWaterproof phone pouches are ideal for outdoor activities such as swimming, snorkeling, kayaking, or hiking in wet conditions. They provide peace of mind knowing that your phone is protected from water, dust, and dirt while still being easily accessible.\n\nWhen choosing a waterproof phone pouch, it is important to consider the size and compatibility with your phone model. Some pouches are designed to fit specific phone sizes, while others are more universal. It is also important to ensure that the p

## SimpleSequentialChain

In [61]:
from langchain_classic.chains import SimpleSequentialChain

In [62]:
llm = ChatOpenAI(temperature=0.9)

# prompt template 1
first_prompt = ChatPromptTemplate.from_template(
    """
    Write detailed information about the product: {product}.

    Include:
    - Color options
    - Product benefits
    - Special features
    """
)

# Chain 1
chain_one = LLMChain(llm=llm, prompt=first_prompt,output_key="product_description")


In [65]:
# prompt template 2
second_prompt = ChatPromptTemplate.from_template(
    """
    Create a product review based on this product description:

    {product_description}
    """
)
# chain 2
chain_two = LLMChain(llm=llm, prompt=second_prompt,output_key="product_review")

In [111]:
overall_simple_chain = SimpleSequentialChain(chains=[chain_one, chain_two],
                                             verbose=True
                                            )

In [67]:
overall_simple_chain.run(product)



> Entering new SimpleSequentialChain chain...
The Luxury Air Mattress is a high-quality sleeping solution that offers comfort and support in a convenient inflatable design. 

Color Options:
The Luxury Air Mattress comes in a sleek and modern black color option, providing a versatile look that can complement any bedroom decor.

Product Benefits:
1. Comfort: The Luxury Air Mattress is designed with advanced air chamber technology that provides customized support and comfort for a restful night's sleep.
2. Durability: Made from durable materials, this air mattress is built to last and can withstand regular use without losing its shape or support.
3. Portability: This air mattress is easy to inflate and deflate, making it a great option for camping trips, guest accommodations, or any situation where a comfortable sleeping surface is needed.
4. Size Options: The Luxury Air Mattress comes in multiple size options, including twin, full, queen, and king, allowing you to choose the perfect si

"I recently purchased the Luxury Air Mattress and I am extremely impressed with its quality and comfort. The advanced air chamber technology provides customized support that allowed me to have a restful night's sleep. The built-in pump made it so easy to inflate and deflate the mattress, saving me time and hassle. \n\nThe waterproof design is a great feature as it makes cleaning and maintenance a breeze. I also love that I can adjust the firmness level to my liking, ensuring a personalized sleeping experience. The raised design of the mattress adds extra comfort and support, making it easy to get in and out of bed.\n\nI also appreciate the multiple size options available, allowing me to choose the perfect size for my needs. The sleek black color option is modern and versatile, complementing any bedroom decor. This air mattress is perfect for camping trips, guest accommodations, or any situation where a comfortable sleeping surface is needed.\n\nOverall, I highly recommend the Luxury Ai

**Repeat the above twice for different products**

In [68]:
product = 'Pillows Insert'
chain.run(product)

from langchain_classic.chains import SimpleSequentialChain

llm = ChatOpenAI(temperature=0.9)

# prompt template 1
first_prompt = ChatPromptTemplate.from_template(
    """
    Write detailed information about the product: {product}.

    Include:
    - Color options
    - Product benefits
    - Special features
    """
)

# Chain 1
chain_one = LLMChain(llm=llm, prompt=first_prompt,output_key="product_description")

# prompt template 2
second_prompt = ChatPromptTemplate.from_template(
    """
    Create a product review based on this product description:

    {product_description}
    """
)
# chain 2
chain_two = LLMChain(llm=llm, prompt=second_prompt,output_key="product_review")

overall_simple_chain = SimpleSequentialChain(chains=[chain_one, chain_two],
                                             verbose=True
                                            )
overall_simple_chain.run(product)



> Entering new SimpleSequentialChain chain...
Pillows Insert:

Color options: Pillows inserts usually come in a neutral white color to easily match with any pillow cover or decor. However, some brands might offer different color options such as grey or beige.

Product benefits: Pillow inserts are essential for maintaining the shape and fluffiness of your pillows. They provide support and comfort for a good night's sleep or are used as decorative pillows on sofas or chairs. Pillow inserts also make it easy to change the look of your living space by simply swapping out the pillow covers.

Special features: Some pillow inserts come with special features such as being hypoallergenic, anti-microbial, and dust mite resistant, making them suitable for individuals with allergies. Additionally, some inserts are made from eco-friendly materials like recycled polyester fiberfill or organic cotton.

Overall, pillow inserts are versatile and practical accessories that enhance the comfort and aest

"I recently purchased a set of pillow inserts and I am incredibly impressed with the quality and functionality of this product. The neutral white color is perfect for matching with any pillow cover or decor, making it versatile for any room in my home.\n\nThe benefits of these pillow inserts are abundant. Not only do they provide essential support and comfort for a restful night's sleep, but they also add a decorative touch to my sofas and chairs. I love how easy it is to change the look of my living space by simply swapping out the pillow covers with these inserts.\n\nWhat sets these pillow inserts apart are their special features. They are hypoallergenic, anti-microbial, and dust mite resistant, making them perfect for individuals like myself who suffer from allergies. I appreciate that they are made from eco-friendly materials, such as recycled polyester fiberfill or organic cotton, as I try to make sustainable choices in my home.\n\nOverall, these pillow inserts have truly enhanced

In [75]:
product = 'Luxury Air Mattress'
chain.run(product)

from langchain_classic.chains import SimpleSequentialChain

llm = ChatOpenAI(temperature=0.9)

# prompt template 1
first_prompt = ChatPromptTemplate.from_template(
    """
    Write detailed information about the product: {product}.

    Include:
    - Color options
    - Product benefits
    - Special features
    """
)

# Chain 1
chain_one = LLMChain(llm=llm, prompt=first_prompt,output_key="product_description")

# prompt template 2
second_prompt = ChatPromptTemplate.from_template(
    """
    Create a product review based on this product description:

    {product_description}
    """
)
# chain 2
chain_two = LLMChain(llm=llm, prompt=second_prompt,output_key="product_review")

overall_simple_chain = SimpleSequentialChain(chains=[chain_one, chain_two],
                                             verbose=True
                                            )
overall_simple_chain.run(product)



> Entering new SimpleSequentialChain chain...
The Luxury Air Mattress is a premium quality mattress designed to provide ultimate comfort and support for a great night's sleep. This air mattress comes in a sleek black color, adding a touch of elegance to any bedroom decor.

Product Benefits:
1. Comfort: The Luxury Air Mattress is constructed with a soft, velvety top layer that provides a plush sleeping surface, ensuring a comfortable and restful night's sleep.
2. Support: The mattress is designed with a sturdy and durable construction that offers firm support, helping to alleviate pressure points and promote proper spinal alignment.
3. Versatility: This air mattress is perfect for both indoor and outdoor use, making it a versatile option for camping trips, overnight guests, or as a temporary bedding solution.
4. Easy Inflation: The mattress includes a built-in electric pump that allows for quick and easy inflation within minutes, saving you time and hassle.

Special Features:
1. Eleva

"I recently purchased the Luxury Air Mattress and I am extremely impressed with the quality and comfort it provides. The soft, velvety top layer feels luxurious to sleep on and ensures a restful night's sleep. I love that the mattress offers firm support, helping to alleviate pressure points and promote proper spinal alignment.\n\nOne of my favorite features is the easy inflation with the built-in electric pump. It only takes a few minutes to fully inflate the mattress, saving me time and hassle. The elevated design is also a great bonus as it provides added insulation from cold surfaces and makes it easier to get in and out of bed.\n\nI appreciate the dual chamber construction that allows for personalized firmness levels for each sleeper, as well as the built-in pillow for added comfort. The puncture-resistant material gives me peace of mind that this mattress will last a long time.\n\nOverall, the Luxury Air Mattress is a top-of-the-line bedding solution that combines luxurious comfo

## SequentialChain

In [96]:
from langchain_classic.chains import SequentialChain

In [97]:
llm = ChatOpenAI(temperature=0.9)


first_prompt = ChatPromptTemplate.from_template(
    """
    Translate the following customer review into English:

    {review}
    """
)

chain_one = LLMChain(llm=llm, prompt=first_prompt,
                     output_key="translated_review"
                    )

In [98]:
second_prompt = ChatPromptTemplate.from_template(
    """Summarize the following review in one concise sentence:

    {translated_review}
    """
)

chain_two = LLMChain(llm=llm, prompt=second_prompt,
                     output_key="summarized_review"
                    )

In [101]:
# prompt template 3: translate to english or other language
third_prompt = ChatPromptTemplate.from_template(
    """
    What language is the following review written in?

    {review}

    Respond with only the language name.
    """
)
# chain 3: input= Review and output= language
chain_three = LLMChain(llm=llm, prompt=third_prompt,
                       output_key="language"
)



In [102]:
# prompt template 4: follow up message that take as inputs the two previous prompts' variables
fourth_prompt = ChatPromptTemplate.from_template(
    """
    Write a follow-up response to the following customer review.

    The response should be written in {language}.

    Review:
    {translated_review}
    """
)
chain_four = LLMChain(llm=llm, prompt=fourth_prompt,
                      output_key="followup_message"
                     )



In [103]:
# overall_chain: input= Review
# and output= English_Review,summary, followup_message
overall_chain = SequentialChain(
    chains=[chain_one, chain_two, chain_three, chain_four],
    input_variables=['review'],
    output_variables=['translated_review', 'summarized_review', 'followup_message'],
    verbose=True
)

In [104]:
review = df.Review[5]
overall_chain(review)



> Entering new SequentialChain chain...

> Finished chain.


{'review': "Je trouve le goût médiocre. La mousse ne tient pas, c'est bizarre. J'achète les mêmes dans le commerce et le goût est bien meilleur...\nVieux lot ou contrefaçon !?",
 'translated_review': "I find the taste mediocre. The foam doesn't hold, it's strange. I buy the same ones in stores and the taste is much better... Old batch or counterfeit!?",
 'summarized_review': "The reviewer finds the taste of the product mediocre and suspects it may be an old batch or counterfeit because the foam doesn't hold and the taste is inferior to the ones bought in stores.",
 'followup_message': "Response:\nNous sommes désolés d'apprendre que vous n'avez pas apprécié le goût de notre produit. Nous prenons la qualité de nos produits très au sérieux et nous vous assurons qu'il n'y a pas de contrefaçon. Il est possible qu'il s'agisse d'un lot défectueux et nous aimerions avoir l'opportunité de rectifier cela pour vous. Veuillez nous contacter pour que nous puissions résoudre ce problème. Merci pour 

**Repeat the above twice for different products or reviews**

In [106]:
review = df.Review[6]
overall_chain(review)



> Entering new SequentialChain chain...

> Finished chain.


{'review': 'Está lu bonita calienta muy rápido, es muy funcional, solo falta ver cuánto dura, solo llevo 3 días en funcionamiento.',
 'translated_review': 'It is very nice, heats up very quickly, it is very functional. Just need to see how long it lasts, I have only been using it for 3 days.',
 'summarized_review': 'Overall positive review highlighting the quick heating and functionality of the product, but concerns about long-term durability due to only 3 days of use.',
 'followup_message': 'Respuesta:\n¡Gracias por tu reseña! Nos alegra saber que estás disfrutando de tu producto y que cumple con tus expectativas en cuanto a velocidad y funcionalidad. Esperamos que te siga proporcionando un buen rendimiento a lo largo del tiempo. ¡Gracias por tu confianza y disfruta tu compra!'}

In [121]:
review = df.Review[1]
overall_chain(review)



> Entering new SequentialChain chain...

> Finished chain.


{'review': 'I loved the waterproof sac, although the opening was made of a hard plastic. I don’t know if that would break easily. But I couldn’t turn my phone on, once it was in the pouch.',
 'translated_review': 'I really liked the waterproof bag, but I was unsure about the durability of the hard plastic opening. Also, I had trouble turning on my phone once it was in the pouch.',
 'summarized_review': 'The waterproof bag has a good design, but there are concerns about the durability of the hard plastic opening and difficulties with accessing the phone while in the pouch.',
 'followup_message': "Response:\nThank you for your feedback! We appreciate your input regarding the durability of the hard plastic opening and the issue with turning on your phone while it's in the pouch. We will definitely take your comments into consideration for future product improvements. If you have any further concerns or need assistance, please don't hesitate to reach out to our customer support team. We va

## Router Chain

In [122]:
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise\
and easy to understand manner. \
When you don't know the answer to a question you admit\
that you don't know.

Here is a question:
{input}"""


math_template = """You are a very good mathematician. \
You are great at answering math questions. \
You are so good because you are able to break down \
hard problems into their component parts,
answer the component parts, and then put them together\
to answer the broader question.

Here is a question:
{input}"""

history_template = """You are a very good historian. \
You have an excellent knowledge of and understanding of people,\
events and contexts from a range of historical periods. \
You have the ability to think, reflect, debate, discuss and \
evaluate the past. You have a respect for historical evidence\
and the ability to make use of it to support your explanations \
and judgements.

Here is a question:
{input}"""


computerscience_template = """ You are a successful computer scientist.\
You have a passion for creativity, collaboration,\
forward-thinking, confidence, strong problem-solving capabilities,\
understanding of theories and algorithms, and excellent communication \
skills. You are great at answering coding questions. \
You are so good because you know how to solve a problem by \
describing the solution in imperative steps \
that a machine can easily interpret and you know how to \
choose a solution that has a good balance between \
time complexity and space complexity.

Here is a question:
{input}"""

biology_template = """You are an excellent biologist. \
You have a deep understanding of living organisms, \
from the molecular and cellular level to entire ecosystems. \
You are skilled at observing patterns in nature, analyzing biological data, \
and explaining complex processes like evolution, genetics, physiology, and ecology. \
You can clearly communicate how life functions and adapts, \
and you make connections between different biological concepts \
to answer challenging questions.

Here is a question:
{input}"""

In [123]:
prompt_infos = [
    {
        "name": "physics",
        "description": "Good for answering questions about physics",
        "prompt_template": physics_template
    },
    {
        "name": "math",
        "description": "Good for answering math questions",
        "prompt_template": math_template
    },
    {
        "name": "History",
        "description": "Good for answering history questions",
        "prompt_template": history_template
    },
    {
        "name": "computer science",
        "description": "Good for answering computer science questions",
        "prompt_template": computerscience_template
    },
    {
        "name": "biology",
        "description": "Good for answering biology questions",
        "prompt_template": biology_template
    }
]

In [124]:
from langchain_classic.chains.router import MultiPromptChain
from langchain_classic.chains.router.llm_router import LLMRouterChain, RouterOutputParser
from langchain_core.prompts import PromptTemplate  # Updated path for prompts (core package)

In [125]:
llm = ChatOpenAI(temperature=0)

In [126]:
destination_chains = {}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    prompt = ChatPromptTemplate.from_template(template=prompt_template)
    chain = LLMChain(llm=llm, prompt=prompt)
    destination_chains[name] = chain

destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)

In [127]:
default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = LLMChain(llm=llm, prompt=default_prompt)

In [128]:
MULTI_PROMPT_ROUTER_TEMPLATE = """Given a raw text input to a \
language model select the model prompt best suited for the input. \
You will be given the names of the available prompts and a \
description of what the prompt is best suited for. \
You may also revise the original input if you think that revising\
it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{{{
    "destination": string \ name of the prompt to use or "DEFAULT"
    "next_inputs": string \ a potentially modified version of the original input
}}}}
```

REMEMBER: "destination" MUST be one of the candidate prompt \
names specified below OR it can be "DEFAULT" if the input is not\
well suited for any of the candidate prompts.
REMEMBER: "next_inputs" can just be the original input \
if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}

<< OUTPUT (remember to include the ```json)>>"""

In [129]:
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destinations_str
)
router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser(),
)

router_chain = LLMRouterChain.from_llm(llm, router_prompt)

In [130]:
chain = MultiPromptChain(router_chain=router_chain,
                         destination_chains=destination_chains,
                         default_chain=default_chain, verbose=True
                        )

In [131]:
chain.run("What is black body radiation?")



> Entering new MultiPromptChain chain...
physics: {'input': 'What is black body radiation?'}
> Finished chain.


"Black body radiation is the electromagnetic radiation emitted by a perfect absorber of radiation, known as a black body. A black body absorbs all radiation that falls on it and emits radiation across the entire electromagnetic spectrum. The spectrum of black body radiation is continuous and depends only on the temperature of the black body. This phenomenon is described by Planck's law, which states that the intensity of radiation emitted by a black body at a given wavelength is proportional to the temperature of the body and the wavelength raised to the fifth power."

In [132]:
chain.run("what is 2 + 2")



> Entering new MultiPromptChain chain...
math: {'input': 'what is 2 + 2'}
> Finished chain.


'The answer to 2 + 2 is 4.'

In [133]:
chain.run("Why does every cell in our body contain DNA?")



> Entering new MultiPromptChain chain...
biology: {'input': 'Why does every cell in our body contain DNA?'}
> Finished chain.


'Every cell in our body contains DNA because DNA is the genetic material that carries the instructions for the development, functioning, and reproduction of all living organisms. DNA contains the information needed to build and maintain an organism, including the proteins that make up our cells and tissues. \n\nHaving DNA in every cell ensures that each cell has the necessary genetic information to carry out its specific functions and to replicate itself accurately during cell division. This ensures that the genetic information is passed on to the next generation of cells. \n\nAdditionally, DNA is constantly being used by cells to carry out processes such as protein synthesis, cell division, and repair. Having DNA in every cell allows for the coordination of these processes and ensures that the organism functions properly as a whole. \n\nIn summary, every cell in our body contains DNA because it is essential for the proper functioning and development of all living organisms.'

**Repeat the above at least once for different inputs and chains executions - Be creative!**

In [140]:
chain.run("How did mathematics and computer graphics work together to create the scientifically accurate black hole in Interstellar?")



> Entering new MultiPromptChain chain...
computer science: {'input': 'How did mathematics and computer graphics work together to create the scientifically accurate black hole in Interstellar?'}
> Finished chain.


"In the movie Interstellar, the scientifically accurate black hole was created using a combination of mathematics and computer graphics. \n\nMathematics played a crucial role in accurately modeling the physics of a black hole. The filmmakers consulted with renowned physicist Kip Thorne, who provided them with equations and simulations that described the behavior of light and gravity near a black hole. These equations were used to create a realistic depiction of the black hole's gravitational lensing effects, as well as the warping of spacetime around it.\n\nComputer graphics were then used to translate these mathematical models into visually stunning images. Advanced rendering techniques were employed to simulate the bending of light around the black hole, creating the iconic visual of the black hole's accretion disk and the distorted background stars.\n\nBy combining the mathematical understanding of black hole physics with cutting-edge computer graphics technology, the filmmakers wer

In [139]:
chain.run("How do matrices make movie characters move and rotate realistically in CGI?")



> Entering new MultiPromptChain chain...
computer science: {'input': 'How do matrices make movie characters move and rotate realistically in CGI?'}
> Finished chain.


"Matrices play a crucial role in creating realistic movement and rotation of movie characters in CGI (Computer Generated Imagery). In CGI, characters are represented as 3D models composed of vertices, edges, and faces. These models are manipulated using matrices to achieve various transformations such as translation, rotation, scaling, and skewing.\n\nTo make a character move and rotate realistically, a series of transformations are applied to the character's model. Each transformation is represented by a matrix that describes how the model should be modified. For example, to move a character from one position to another, a translation matrix is used to shift the model along the x, y, and z axes. Similarly, rotation matrices are used to rotate the character around a specific axis.\n\nBy combining multiple transformation matrices, complex movements and rotations can be achieved, giving the character a lifelike appearance. Additionally, matrices are used to calculate lighting effects, sh